# PyTorch Workflow

Let's explore an end to end workflow

In [ ]:
what_were_covering = {1:'data (prepare and load)',
                      2:'build a model',
                      3:'fit the model',
                      4:'maing predictions',
                      5:'saving and loading a model',
                      6:'put it all together'}

In [ ]:
what_were_covering

In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt

#Check Pytorch version
torch.__version__

## 1. Data Preparing and Loading

In [ ]:
# Create *known* parameters
weight = 0.7
bias = 0.3

# Create 
start = 0
end = 1
step = 0.02
X = torch.arange(start,end,step).unsqueeze(dim=1) 
y = weight*X + bias

X[:10], y[:10]

# Creating test and train split

In [ ]:
train_split = int(0.8*len(X))
X_train, y_train = X[:train_split], y[:train_split]
X_test, y_test = X[train_split:], y[train_split:]

len(X_train), len(y_train), len(X_test), len(y_test)

How might we better visualize our data?

"Visualize, visualize, visualize"

In [ ]:
X_train, y_train

In [ ]:
def plot_predictions(train_data=X_train,
                     train_labels = y_train,
                     test_data=X_test,
                     test_labels = y_test,
                     predictions = None):
    plt.figure(figsize=(10,7))

    plt.scatter(train_data,train_labels, c="b",
                s=4, label="Training data")
    plt.scatter(test_data,test_labels, c="g", s=4,
                label="Testing data")
    if predictions is not None:
        plt.scatter(test_data,predictions, c="r",
                    s=4, label="Predictions")
    plt.legend(prop={"size":14})

In [ ]:
plot_predictions()

In [ ]:
# Build a model
# Our model starts with random values (weight & bias) then updates values to represent real values overtime
# Uses gradient descent and backpropagation
class LinearRegressionModel(nn.Module): #<- almost everything inherits from nn.module
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(1,
                                                requires_grad = True,
                                                dtype=torch.float32))
        self.bias = nn.Parameter(torch.randn(1,
                                             requires_grad = True,
                                             dtype = torch.float32))
    def forward(self, x:torch.Tensor) -> torch.Tensor:
        return self.weights *x + self.bias
        


### PyTorch Model building essentials

* torch.nn - contains all of the buildings for computational graphs ( a neural network can be considered a computational graph)
* torch.nn.Parameter - what parameters should our model try and learn
* torch.nn.Module - The base class for all neural network modules, if you subclass it, you should overrite forward()
* torch.optim - this is where the optimizers in PyTorch live, they will help with gradient descent
* def forward() - All nn.Module subclasses require you to overwrite forward(), this method defines what happens in the forward computation

### Checking the contents of our PyTorch model



In [ ]:
# Create a random seed
torch.manual_seed(42)

# Create an instance of the model (this is a sublcass of nn.module)
model_0 = LinearRegressionModel()

# Check out the parameters
list(model_0.parameters())

In [ ]:
# List named parameters
model_0.state_dict()

In [ ]:
# Making predictions
with torch.inference_mode():
    y_preds = model_0(X_test)
y_preds

In [ ]:
plot_predictions(predictions=y_preds)

In [ ]:
# Train the model:

#First need to use a loss function to evaluate output
list(model_0.parameters())


In [ ]:
# Setup a loss function
loss_fn = nn.L1Loss()

# Setup an optimizer
optimizer = torch.optim.SGD (params = model_0.parameters(),
                             lr = 0.01) # lr = learning rate = possibly the most important hyperparameter

### Building a training loop (and a testing loop) in PyTorch

A couple of things we need:
0. Loop through the data
1. Forward pass( this involves data moving through our model's `forward()` functions to make predictions on data - forward propagation)
2. calculate the loss
3. Optimizer zero grad
4. loss backward - move backwards through the network to calculate the gradients of each of the parameters of our model with respect to loss
5. Optimizer step


In [ ]:
# an epoch is one loop through the data...
epochs = 200

epoch_count = []
loss_values = []
test_loss_values = []
### Training
# 0. Loop through the data
for epoch in range(epochs):
    #Set the model to training mode
    model_0.train() # train mode in PyTorch Set all parameters to 

    # 1.Forward pass
    y_pred = model_0(X_train)

    # 2. Calculate the loss
    loss = loss_fn(y_pred,y_train)
    # print(f"loss:{loss}")

    #3 Optimizer zero grad
    optimizer.zero_grad()

    # 4. Backpropagation
    loss.backward()

    # 5. Step the optimizer
    optimizer.step()

    ### Testing
    model_0.eval() #Turns off different settings in the model not needed for evaluation/testing (dropout/batchnorm layers)
    with torch.inference_mode(): # turns off gradient tracking and a couple more things behind the scenes
        # 1. Do the forward pass
        test_pred = model_0(X_test)

        # 2. Calculate the loss
        test_loss = loss_fn(test_pred, y_test)

    if epoch % 10 == 0:
        epoch_count.append(epoch)
        loss_values.append(loss)
        test_loss_values.append(test_loss)
        print(f"Epoch: {epoch} | Loss: {loss} | Test loss: {test_loss}")


    # # Print out model _state_dict()
    # print(model_0.state_dict())

In [ ]:
with torch.inference_mode():
    y_preds_new = model_0(X_test)

In [ ]:
plot_predictions(predictions=y_preds_new)

In [ ]:
model_0.state_dict()

In [ ]:
import numpy as np
plt.plot(epoch_count, torch.tensor(loss_values).detach().numpy(), label = "Train loss")
plt.plot(epoch_count, torch.tensor(test_loss_values).detach().numpy(), label = "Test loss")
plt.title("training and test loss curves")
plt.ylabel("Loss")
plt.xlabel("Epochs")
plt.legend();

## Saving a model in PyTorch

There are three main methods to save and load models
1. torch.save() - saves model in pickle format
2. torch.load() - loads saved models
3. torch.nn.Module.load_state_dict() - this allows us to load a model's saved state dictionary 

In [ ]:
# Saving our PyTorch model
from pathlib import Path

# 1. Create models directory
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# 2. Create model save path
MODEL_NAME = "01_pytorch_workflow_model_0.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

torch.save(obj=model_0.state_dict(),f= MODEL_SAVE_PATH)

In [ ]:
!ls -l models

In [ ]:
## Loading a PyTorch model
loaded_model_0 = LinearRegressionModel()

#Load the saved state dict of model_0

loaded_model_0.load_state_dict(torch.load(MODEL_SAVE_PATH))

In [ ]:
loaded_model_0.state_dict()

In [ ]:
# Make some predictions
loaded_model_0.eval()
with torch.inference_mode():
    loaded_model_preds = loaded_model_0(X_test)

loaded_model_preds

In [ ]:
# Make some predictions
model_0.eval()
with torch.inference_mode():
    model_preds = model_0(X_test)

model_preds

In [ ]:
## Putting it all together
#device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Create some data using the linear regression formula 
weight = 0.7
bias = 0.3

# create a range 
start = 0
end = 1
step = 0.02

#create X and Y
X = torch.arange(start, end, step).unsqueeze(dim=1)
y = weight * X + bias
X[:10], y[:10]

In [ ]:
#split data
train_split = int(0.8*len(X))
X_train, y_train = X[:train_split], y[:train_split]
X_test, y_test = X[train_split:], y[train_split:]
len(X_train), len(y_train), len(X_test), len(y_test)

In [ ]:
plot_predictions(X_train,y_train,X_test, y_test)

In [ ]:
## 6.2 Building a PyTorch Linear Model
class LinearRegressionModelv2(nn.Module):
    def __init__(self):
        super().__init__()
        # Uuse nn.Linear() for creating the model parameters
        self.linear_layer = nn.Linear(in_features=1,
                                      out_features=1)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear_layer(x)
# Set the manual seed
torch.manual_seed(42)
model_1 = LinearRegressionModelv2()
model_1, model_1.state_dict()

### 6.3 Training

We need
- Loss function
- Optimizer
- Training loop
- Testing loop

In [ ]:
# Set the model to use the target device
next(model_1.parameters()).device

loss_fn = nn.L1Loss()

optimizer = torch.optim.SGD(params = model_1.parameters(),
                            lr=0.01)

In [ ]:
torch.manual_seed(42)

epochs = 200

# Put data on the same device
X_train = X_train.to(device)
y_train = y_train.to(device)
X_test = X_test.to(device)
y_test = y_test.to(device)

In [ ]:

for epoch in range(epochs):
    model_1.train()

    #forward pass
    y_pred = model_1(X_train)

    #Calcualte loss
    loss = loss_fn(y_pred, y_train)

    # Optimizer zero grad
    optimizer.zero_grad()

    # Perform backpropagation
    loss.backward()

    # Optimizer step
    optimizer.step()

    ### Testing
    model_1.eval()
    with torch.inference_mode():
        test_pred = model_1(X_test)
        test_loss = loss_fn(test_pred, y_test)

    if epoch % 10 == 0:
        print(f"Epoch: {epoch} | Loss: {"loss"}")

In [ ]:
model_1.state_dict()

In [ ]:
# Making and evaluating predictions
with torch.inference_mode():
    y_preds = model_1(X_test)
y_preds

In [ ]:
plot_predictions(predictions=y_preds)

In [ ]:
### Save and load a model
from pathlib import Path

#1. Create models directory
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

#2. Create model save path
MODEL_NAME = "01_pytorch_workflow_model_1.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

# 3. Save the model state dict
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model_1.state_dict(),
           f = MODEL_SAVE_PATH)

In [ ]:
# Load a PyTorch model
loaded_model_1 = LinearRegressionModelv2()

loaded_model_1.load_state_dict(torch.load(MODEL_SAVE_PATH))

In [ ]:
loaded_model_1.to(device)

In [ ]:
loaded_model_1.state_dict()

In [ ]:
loaded_model_1.eval()
with torch.inference_mode():
    loaded_model_1_preds = loaded_model_1(X_test)


